In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path
import os
import flap

sys.path.insert(0, str(Path.cwd().parent))
from neuro_bes import data

In [ ]:
shot='20250225.054'
path='/data2/W7-X/processed_data/APDCAM/flap_recon/'
try:
    file_list=os.listdir(os.path.join(path,shot))
except FileNotFoundError:
    raise ValueError("Directory " + os.path.join(path,shot) + " does not exist")


file_name_light=[i for i in file_list if ('light_ds_orig' in i)]
file_name_density=[i for i in file_list if ('dens' in i)]
file_name_light_recon=[i for i in file_list if ('light_recon' in i)]
if len(file_name_light)>1:
    raise ValueError("There is more than one data source for shot " + shot)
if len(file_name_light)==0:
    raise ValueError("Data source not found for shot " + shot)
file_name_light=file_name_light[0]
file_name_density=file_name_density[0]
file_name_light_recon=file_name_light_recon[0]

In [ ]:
file_name_light

In [ ]:
light=flap.load(os.path.join(os.path.join(path,shot),file_name_light))
density=flap.load(os.path.join(os.path.join(path,shot),file_name_density))
light_recon=flap.load(os.path.join(os.path.join(path,shot),file_name_light_recon))

In [ ]:
time_instances_light_recon=light_recon.coordinate('Time')[0][:,0]
time_instances_density=density.coordinate('Time')[0][:,0]
time_instances_light_recon==time_instances_density

In [ ]:
avg_timedelta=np.mean(np.diff(time_instances_density))
if avg_timedelta<0.01:
    raise ValueError(f"Downsampling frequency is above limit ({int(1/avg_timedelta)} Hz) in shot " + shot)

In [ ]:
avg_timedelta

In [ ]:
time_instances_light=light.coordinate('Time')[0][:,0]
mapping = {val: idx for idx, val in enumerate(time_instances_light)}
mask_timeinstance = np.array([mapping[val] for val in time_instances_density])
light_data=light.data[mask_timeinstance,:]

In [ ]:
density_data=density.data
light_recon_data=light_recon.data

In [ ]:
r_coord=light.coordinate('Device R')[0][mask_timeinstance[0]]
mask_samegrid_1=np.all(light.coordinate('Device R')[0]==r_coord,axis=1)[mask_timeinstance]
mask_samegrid_2=np.all(density.coordinate('Device R')[0]==r_coord,axis=1)
light_data=light_data[mask_samegrid_1*mask_samegrid_2]
density_data=density_data[mask_samegrid_1*mask_samegrid_2]
light_recon_data=light_recon_data[mask_samegrid_1*mask_samegrid_2]

In [ ]:
mask_goodrecon=np.sqrt(np.mean((light_data-light_recon_data)**2,axis=1))/np.mean(light_data,axis=1)<0.1
light_data=light_data[mask_goodrecon]
density_data=density_data[mask_goodrecon]
light_recon_data=light_recon_data[mask_goodrecon]
time_instances_density=time_instances_density[mask_goodrecon]

In [ ]:
mask_goodrecon.shape

In [ ]:
grid=r_coord
energy=0
species="Na"
ID="we_"+shot
zeff=0
q=0
temperature=np.array(0)
verbose="W7X experimental data shot "+shot+", 10Hz averaging, curated for good SPADE recons"
tags=['Time instance ' + str(i) + ' s' for i in time_instances_density]

In [ ]:
grid

In [ ]:
test_data=data.besInferenceDatapoints(grid=grid,energy=energy,species=species,ID=ID,zeff=zeff,q=q,temperature=temperature,verbose=verbose)

In [ ]:
test_data.add_datapoints_bulk(density_data, light_data, tags)

In [ ]:
density_data.shape

In [ ]:
plt.plot(test_data.grid,test_data.emissions[10])
plt.plot(test_data.grid,light_recon_data[10])

In [ ]:
#test_data.export_to_h5(path_to_dir="/home/molnarbalazs/data/BES_ML_modelling")